In [21]:
import os

print(os.getcwd())

/Users/mac


In [23]:
os.listdir()

['.Rhistory',
 'Untitled7.ipynb',
 'Epilepsy_Detection_ML_Final_one_RF_LR_LDA_FIXED.ipynb',
 'penguins_missing_values.jpeg',
 '.config',
 'Music',
 'Epilepsy_LabConfirmed_Final.ipynb',
 'Epilepsy_Detection_ML_Final_StudentSummary.ipynb',
 '.condarc',
 'Bayesian_regression_model_defining_log_likelihood.rds',
 'Bayesian_regression_model_WAIC.rds',
 'Untitled5.ipynb',
 'Untitled1.ipynb',
 '.DS_Store',
 'Bayesian_regression_model_WAIC_quadratic.stan',
 'Epilepsy_Detection_ML_Final_CMFixed.ipynb',
 '.CFUserTextEncoding',
 'Bayesian_regression_model_WAIC_quadratic.rds',
 '.xonshrc',
 'Lab5_.ipynb',
 'Untitled3.ipynb',
 'Untitled.ipynb',
 '.zshrc',
 'Untitled4.ipynb',
 'Bayesian_regression_model.rds',
 '.local',
 'Untitled6.ipynb',
 'Epilepsy_Detection_ML_Final_one_WithModelComparison.ipynb',
 'Pictures',
 'Epilepsy_.ipynb',
 'Epilepsy_Detection_Complete_Coursework.ipynb',
 'Epilepsy_Detection_ML_Final_WithOptimization.ipynb',
 'Epilepsy_Detection_ML_Final_one_WithExpandedModels.ipynb',
 'ran

In [25]:
import pandas as pd

df = pd.read_csv(
    "Desktop/Clinical-Readmission-Risk-Analytics/data/diabetic_data_processed.csv"
)

print(df.shape)

(101766, 49)


In [27]:
df_model = df.copy()

In [29]:
df_model.columns[:10]

Index(['race', 'gender', 'age', 'admission_type_id',
       'discharge_disposition_id', 'admission_source_id', 'time_in_hospital',
       'payer_code', 'medical_specialty', 'num_lab_procedures'],
      dtype='object')

In [31]:
'readmitted_binary' in df_model.columns

True

In [33]:
df_model['readmitted_binary'].value_counts()

readmitted_binary
0    90409
1    11357
Name: count, dtype: int64

In [35]:
df_model.select_dtypes(include='object').columns

Index(['race', 'gender', 'age', 'payer_code', 'medical_specialty', 'diag_1',
       'diag_2', 'diag_3', 'metformin', 'repaglinide', 'nateglinide',
       'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide',
       'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
       'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton',
       'insulin', 'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted',
       'diag_1_cat', 'diag_2_cat', 'diag_3_cat'],
      dtype='object')

In [37]:
len(df_model.select_dtypes(include='object').columns)

37

In [41]:
df_model = df_model.drop(
    columns=[
        'readmitted',
        'diag_1',
        'diag_2',
        'diag_3'
    ]
)

print(df_model.shape)

(101766, 45)


In [43]:
[col for col in ['readmitted','diag_1','diag_2','diag_3']
 if col in df_model.columns]

[]

In [45]:
categorical_cols = df_model.select_dtypes(include='object').columns

print("Categorical columns:", len(categorical_cols))

Categorical columns: 33


In [47]:
df_model_encoded = pd.get_dummies(
    df_model,
    columns=categorical_cols,
    drop_first=True
)

print(df_model_encoded.shape)

(101766, 194)


In [49]:
X = df_model_encoded.drop('readmitted_binary', axis=1)

y = df_model_encoded['readmitted_binary']

print(X.shape)
print(y.shape)

(101766, 193)
(101766,)


In [51]:
print(y.value_counts(normalize=True) * 100)

readmitted_binary
0    88.840084
1    11.159916
Name: proportion, dtype: float64


In [53]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(81412, 193)
(20354, 193)


In [55]:
print(y_train.value_counts(normalize=True) * 100)

print("\nTest Set")

print(y_test.value_counts(normalize=True) * 100)

readmitted_binary
0    88.839483
1    11.160517
Name: proportion, dtype: float64

Test Set
readmitted_binary
0    88.842488
1    11.157512
Name: proportion, dtype: float64


In [57]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

lr_model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [59]:
y_pred = lr_model.predict(X_test)

print(y_pred[:10])

[0 0 0 0 0 0 0 0 0 0]


In [61]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.89      1.00      0.94     18083
           1       0.48      0.01      0.02      2271

    accuracy                           0.89     20354
   macro avg       0.69      0.51      0.48     20354
weighted avg       0.84      0.89      0.84     20354



In [63]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[18053    30]
 [ 2243    28]]


In [65]:
from sklearn.linear_model import LogisticRegression

lr_balanced = LogisticRegression(
    class_weight='balanced',
    max_iter=2000,
    random_state=42
)

lr_balanced.fit(X_train, y_train)

print("Balanced Logistic Regression trained!")

Balanced Logistic Regression trained!


/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [67]:
y_pred_balanced = lr_balanced.predict(X_test)

In [69]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_balanced))

              precision    recall  f1-score   support

           0       0.92      0.67      0.77     18083
           1       0.17      0.55      0.26      2271

    accuracy                           0.65     20354
   macro avg       0.55      0.61      0.52     20354
weighted avg       0.84      0.65      0.72     20354



In [71]:
from sklearn.metrics import confusion_matrix

cm_balanced = confusion_matrix(y_test, y_pred_balanced)

print(cm_balanced)

[[12051  6032]
 [ 1018  1253]]


In [73]:
from sklearn.metrics import roc_auc_score

y_prob = lr_balanced.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_prob)

print("ROC-AUC:", auc)

ROC-AUC: 0.6518140226875472


In [75]:
import joblib

joblib.dump(X_train, "X_train.pkl")
joblib.dump(X_test, "X_test.pkl")
joblib.dump(y_train, "y_train.pkl")
joblib.dump(y_test, "y_test.pkl")

print("Train/Test data saved!")

Train/Test data saved!
